# Step 4 — The Contestants: Three Ways to Answer the Quiz

We have a vetted quiz about *very recent* news (Step 3). Now we line up the
LLM contestants and ask the tutorial's central question: **where does an
LLM's knowledge actually come from?** We test each model three ways:

| Method | What the model gets | What it measures |
|---|---|---|
| `closed_book` | question + options, nothing else | what's in the weights |
| `web_search` | same prompt, search tool ON | what retrieval adds |
| `debate` | 3 copies of the model argue over 3 rounds (GPT models only) | what deliberation adds |

The questions were written from articles published *after* these models'
training data was collected — so closed-book answers must come from stale
weights or lucky guessing, and the gap between conditions is the story.

The pipeline is three scripts, same shape as Step 3:

| Stage | Script | Output |
|---|---|---|
| Closed book / web search (once per method × model) | `scripts/04-1_generate_answers.py` | `data/answers/answers_<method>_<model>.jsonl` |
| Debate (once per GPT model) | `scripts/04-2_generate_debate_answers.py` | `data/answers/answers_debate_<model>.jsonl` |
| Merge to tidy CSV | `scripts/04-3_combine_answers.py` | `data/answers/answers_combined.csv` |

## 1. What the contestant sees

Almost nothing. Unlike the judge (who got the full article), a contestant
gets the question and four lettered options — **never the article**. If we
leaked the article, every method would score 100% and measure only reading
comprehension:

In [1]:
from toolkit import prompts
from toolkit.utils import load_jsonl

selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]

user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)
print(user_prompt)

QUESTION:
What local target for net zero carbon emissions did Andy Burnham set while serving as mayor of Greater Manchester?

OPTIONS:
A. 2030
B. 2035
C. 2038
D. 2050



The system half sets the role and — crucially — forces a commitment: pick
exactly one letter, always, plus a confidence between 0 and 1. Refusals and
"it depends" answers would be ungradable:

In [2]:
print(prompts.ANSWER_SYSTEM_PROMPT)

You are an expert news-quiz contestant. Each question was written from a
recently published news article (within the last few weeks). You are NOT
given the article — answer from what you know or can find.

Rules:
- Pick the single best option: exactly one of A, B, C, or D.
- Always commit to one letter, even if you are unsure.
- Give 1-2 sentences of reasoning and a confidence between 0 and 1.



## 2. Method 1 — closed book

One structured-output call, the same machinery as Steps 2–3. The Pydantic
schema is the whole grading contract:

```python
class Answer(BaseModel):
    answer_letter: Literal["A", "B", "C", "D"]
    confidence: float          # 0-1, self-reported
    reasoning: str             # 1-2 sentences
```

`answer_question()` makes the call and grades it on the spot
(`is_correct = answer_letter == correct_letter`):

In [3]:
from toolkit.answers import answer_question

closed = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="closed_book"
)

print(question["question"], "\n")
for letter, option in zip("ABCD", question["options"]):
    mark = "*" if letter == question["correct_letter"] else " "
    print(f"  {mark}{letter}. {option}")
print(f"\nanswered {closed['answer_letter']} "
      f"(confidence {closed['confidence']:.2f}) -> "
      f"{'CORRECT' if closed['is_correct'] else 'WRONG'}")
print("WHY:", closed["reasoning"])

What local target for net zero carbon emissions did Andy Burnham set while serving as mayor of Greater Manchester? 

   A. 2030
   B. 2035
  *C. 2038
   D. 2050

answered D (confidence 0.92) -> WRONG
WHY: Andy Burnham set Greater Manchester’s net zero carbon target for 2038, which is earlier than the UK’s national 2050 goal. That date has been a key part of his mayoral climate agenda.


## 3. Method 2 — the same question, with web search

One flag changes: `use_web_search=True` hands the model the provider's
built-in search tool (OpenAI's `web_search`, Gemini's `google_search`).
The prompt is **byte-for-byte identical** — that's deliberate. When two
conditions differ by exactly one thing, the accuracy gap *is* the effect
of that thing.

The raw API response also tells us whether the model actually searched
(`search_used`) — models sometimes answer from weights even when handed
a search tool:

In [4]:
web = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="web_search"
)

print(f"closed_book : {closed['answer_letter']} "
      f"(confidence {closed['confidence']:.2f})")
print(f"web_search  : {web['answer_letter']} "
      f"(confidence {web['confidence']:.2f}, "
      f"searched: {web['search_used']})")
print("WHY:", web["reasoning"])

closed_book : D (confidence 0.92)
web_search  : C (confidence 0.99, searched: True)
WHY: Andy Burnham set Greater Manchester’s carbon-neutral/net-zero local target for 2038. That matches the city-region’s official ambition and the mayor’s stated goal.


## 4. Method 3 — a society of minds

The debate method asks: can a model do better by *arguing with itself*?
Three copies of the same model:

1. **Round 0** — each agent answers independently (closed book).
2. **Rounds 1–2** — each agent sees the *other* agents' answers and
   reasoning, then answers again, revised or not.
3. **Verdict** — majority vote. A 1-1-1 tie goes to the letter with the
   highest mean confidence, then alphabetically — deterministic, so the
   same transcript always grades the same way.

This is the "society of minds" setup (Du et al. 2023), and it introduces
one new tool: the **openai-agents SDK**. Each debater is an `Agent` — a
role (our same system prompt), a model, and an `output_type` (our same
`Answer` schema, which the SDK validates) — executed with `Runner.run`:

```python
from agents import Agent, Runner

debater = Agent(
    name="Debater",
    model="gpt-5.4-mini-2026-03-17",
    instructions=prompts.ANSWER_SYSTEM_PROMPT,
    output_type=Answer,            # SDK-validated structured output
)
result = await Runner.run(debater, user_prompt)
result.final_output                # -> an Answer instance
```

Two consequences of that choice. The SDK targets OpenAI's Responses API,
so **only the four GPT models debate** — Gemini sits this condition out.
And the SDK is async-native, so the three agents in a round run
*concurrently* (`asyncio.gather`) instead of one after another. The round
structure itself is still a plain Python loop — a debate's turn order is
fixed, so there is nothing for LLM-driven handoffs to decide.

It costs **9 LLM calls per question** (3 agents × 3 rounds), so watch the
meter. One async wrinkle: Jupyter already runs an event loop, so we
`await` the async variant directly (IPython supports top-level `await`;
plain scripts use the sync `debate_question` instead):

In [5]:
from toolkit.debate import debate_question_async

debate = await debate_question_async(question, model="gpt-5.4-mini-2026-03-17")

for r, round_entries in enumerate(debate["debate"]["transcript"]):
    label = "independent" if r == 0 else f"revision {r}"
    print(f"--- round {r} ({label}) ---")
    for e in round_entries:
        print(f"  agent {e['agent']}: {e['answer_letter']} "
              f"({e['confidence']:.2f}) {e['reasoning'][:80]}")
print(f"\nvotes {debate['debate']['vote_counts']} -> "
      f"final {debate['answer_letter']} "
      f"({'CORRECT' if debate['is_correct'] else 'WRONG'})")

--- round 0 (independent) ---
  agent 0: D (0.93) Andy Burnham’s Greater Manchester mayoralty set a target of reaching net zero ca
  agent 1: A (0.86) Andy Burnham has pushed Greater Manchester to reach net zero carbon emissions by
  agent 2: A (0.93) Andy Burnham set Greater Manchester’s target to reach net zero carbon emissions 
--- round 1 (revision 1) ---
  agent 0: C (0.97) Andy Burnham set Greater Manchester’s net zero target for 2038. That matches the
  agent 1: C (0.99) Andy Burnham set Greater Manchester’s net zero target for 2038 while mayor. That
  agent 2: C (0.98) Andy Burnham set Greater Manchester’s net zero target for 2038. That matches the
--- round 2 (revision 2) ---
  agent 0: C (0.99) Andy Burnham set Greater Manchester’s local net zero target at 2038 while mayor.
  agent 1: C (0.99) Andy Burnham set Greater Manchester’s net zero carbon emissions target for 2038.
  agent 2: C (0.99) Andy Burnham set Greater Manchester’s net zero target at 2038, which is the loca

vo

## 5. The full experiment, from the command line

Answering 100 questions at scale is script work. One run = one method ×
one model = one JSONL file, with the same crash-safe append + resume
behavior as every other step:

```bash
# 04-1: closed book + web search, all six models (1,200 calls)
for METHOD in closed_book web_search; do
  for M in gpt-5.4-mini-2026-03-17 gpt-5.5-2026-04-23 gpt-5.6-luna \
           gpt-5.6-terra gemini-3.1-flash-lite gemini-3.5-flash; do
    uv run python scripts/04-1_generate_answers.py \
        --model $M --method $METHOD --parallel
  done
done

# 04-2: debates, the four GPT models (3,600 calls — 9 per question)
for M in gpt-5.4-mini-2026-03-17 gpt-5.5-2026-04-23 \
         gpt-5.6-luna gpt-5.6-terra; do
  uv run python scripts/04-2_generate_debate_answers.py \
      --model $M --parallel
done

# 04-3: merge every per-run file into one tidy CSV
uv run python scripts/04-3_combine_answers.py \
    --input-dir data/answers --glob 'answers_*.jsonl'
```


## 6. The map

| This notebook | Where it lives |
|---|---|
| §1 contestant prompts | `toolkit.prompts.ANSWER_SYSTEM_PROMPT`, `build_answer_user_prompt()` |
| §2–3 one answer | `toolkit.answers.Answer`, `answer_question()`, `_detect_search_use()` |
| §4 debate | `toolkit.debate` (openai-agents SDK): `debate_question()`, `_aggregate_votes()`; prompts in `DEBATE_REVISION_SYSTEM_PROMPT`, `build_debate_revision_prompt()` |
| §5 at scale | `scripts/04-1_generate_answers.py` (per method × model) + `scripts/04-2_generate_debate_answers.py` (per GPT model); `toolkit.answers.answer_questions()` / `toolkit.debate.debate_questions()`; merged by `scripts/04-3_combine_answers.py` |

---

### Next up 📊

Run the sweep above, then open
[`04_answer_analysis.ipynb`](../analysis/04_answer_analysis.ipynb) for the
leaderboard: accuracy by method, what search buys, and what the debates
changed.
